## PBL Real Project - Married or not

1. https://agtechlab.pythonanywhere.com/ 에 접속하여 회원가입해 주세요. (비밀번호는 단순하게 만드는 것을 권장. 예: 1234)
2. `username` 에 이메일 형식의 아이디를 기입해 주세요.
3. `password` 에 비밀번호를 기입해 주세요.

In [ ]:
project = "married"  # 수정하지 마세요
username = ""  # 회원가입 시 사용한 이메일아이디 (예시. abc@hello.com)
password = ""  # 비밀번호

리더보드 제출을 위한 기본 설정: 아래 코드를 실행해주세요.


In [ ]:
import os
import urllib.request

if not os.path.exists("competition_married.py"):
    url = "https://raw.githubusercontent.com/agtechresearch/LectureMLbasic/main/competition/competition_married.py"
    filename = "competition_married.py"
    urllib.request.urlretrieve(url, filename)

아래 코드를 실행하여 데이터를 다운로드 받습니다: 3개의 csv 파일이 datasetmarried 폴더에 다운로드됨

 * dataset.csv: 과거 결혼매칭 데이터 -> 학습에 사용할 데이터셋
 * problem.csv: ML 모델에 의하여 결혼 예측을 수행하여야 할 데이터셋
 * submission.csv: 리더보드 서버 제출을 위한 파일 형식


In [ ]:
import competition_married as competition

# 파일 다운로드
competition.download_competition_files(project)

Colab 환경에서 한글 폰트 설정을 위한 라이브러리 설치 (한글이 깨져서 나타날 경우 설치 필요)

In [ ]:
!pip install koreanize-matplotlib

----------

## 1. 데이터 불러오기

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# 경고 무시
warnings.filterwarnings("ignore")

# Data 경로 설정
DATA_DIR = "datasetmarried"

In [ ]:
# 학습에 사용할 data set 로드 (dataset.csv)
dataset = pd.read_csv(os.path.join(DATA_DIR, "dataset.csv"))

# problem set 로드 (problem.csv)
problemset = pd.read_csv(os.path.join(DATA_DIR, "problem.csv"))

----------

## 2. 데이터셋 확인

▶ 전반적인 데이터셋 정보 확인 (**.info()**) <br>
> 데이터셋의 **컬럼정보와 값의 수, 각 컬럼별 데이터타입, 행의 갯수 등**의 정보 <br>

In [ ]:
# 데이터셋 정보 확인
dataset.info()

----------

## 3. 전처리: 결측치 처리

In [ ]:
# 항목별 결측치 비율 확인
dataset.isna().mean()

In [ ]:
# 결측치 처리
dataset = dataset.dropna(axis=0)
dataset

In [ ]:
# object 타입의 문자열 변수를 숫자형으로 변환
dataset = pd.get_dummies(dataset, columns=['gender'], drop_first=True)

In [ ]:
dataset.head(5)

----------

## 4. 모델 학습 및 평가


In [ ]:
# 데이터셋, 독립변수와 종속변수 분리: 독립변수 -> x, 종속변수 -> y
x = dataset.drop(['married'], axis=1)
y = dataset['married']

In [ ]:
# train set 과 test set 으로 데이터를 나누기 위해 train_test_split 함수를 불러옴
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.3, random_state=100)

In [ ]:
# K-최근접 이웃 (K-Nearest Neighbors - KNN) 라이브러리 import
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier(n_neighbors=5)
model.fit(x_train, y_train)

In [ ]:
from sklearn.metrics import accuracy_score

pred = model.predict(x_test)  # 테스트 데이터로 예측
print("Test accuracy :", accuracy_score(y_test, pred))  # 정확도

## Problem set 문제에 대한 결혼유무 예측 및 리더보드 결과 제출

- 아래 제출 프로세스가 느리다고 중지 후 다시 코드를 여러차례 재실행하는 경우 패널티가 발생할 수 있습니다. (제출 과정에서 제출 횟수 이슈 발생 가능: 하루 최대 200회 까지 가능)
- 제출에 성공할 경우, "제출에 성공하였습니다"의 메세지와 함께 제출 결과 accuracy 가 화면에 출력됩니다.
- 제출결과는 또한 [대회 페이지(리더보드 서버)](https://agtechlab.pythonanywhere.com/competitions/married/)의 `리더보드` 와 `제출` 탭에서 확인할 수 있습니다.


In [ ]:
# object 타입의 문자열 변수를 숫자형으로 변환: 윗부분 학습에 사용된 데이터와 동일하게 변환
problemset = pd.get_dummies(problemset, columns=['gender'], drop_first=True)

In [ ]:
# 문제 데이터(problemset)에 대한 예측값을 구함
problem_pred = model.predict(problemset)

# 리더보드 서버 제출을 위한 파일 생성
submission = pd.read_csv(os.path.join(DATA_DIR, "submission.csv"))
submission["married"] = problem_pred

# 제출
competition.submit(project, username, password, submission)